In [ ]:
from pathlib import Path
import json
import polars as pl

projectPath = Path.cwd().resolve()
while projectPath.parent != projectPath and not (projectPath / 'Data' / 'data').exists():
    projectPath = projectPath.parent
assert (projectPath / 'Data' / 'data').exists()
dataPath = projectPath / 'Data' / 'data'
outputPath = projectPath / 'pipelines' / 'dataprep' / 'output'
outputPath.mkdir(parents=True, exist_ok=True)
toCamelCase = lambda value: value.split('_')[0] + ''.join(part[:1].upper() + part[1:] for part in value.split('_')[1:])
readCsv = lambda fileName: pl.read_csv(dataPath / fileName, try_parse_dates=True).rename(lambda columnName: toCamelCase(columnName))
mlTable = readCsv('ml_table.csv')
matchedTable = readCsv('matched.csv')
s1Observations = readCsv('s1-observations.csv')
s2Observations = readCsv('s2-observations.csv')
showcaseApply = readCsv('showcase_apply.csv')
splits = readCsv('splits-v1.csv')
featureGroupsRaw = json.loads((dataPath / 'features.json').read_text(encoding='utf-8'))
featureGroups = {groupName: list(map(toCamelCase, columns)) for groupName, columns in featureGroupsRaw.items()}
mlTable.shape, matchedTable.shape, s1Observations.shape, s2Observations.shape, showcaseApply.shape, splits.shape


In [ ]:
featureSetGroups = {
    'F0': ['F0'],
    'F0+F1': ['F0', 'F1'],
    'F0+F1+F2': ['F0', 'F1', 'F2'],
    'F2': ['F2']
}
featureSetColumns = {
    featureSet: list(dict.fromkeys(sum(map(lambda groupName: featureGroups[groupName], groupNames), [])))
    for featureSet, groupNames in featureSetGroups.items()
}
featureSets = pl.DataFrame([
    {
        'featureSet': featureSet,
        'featureGroup': groupName,
        'featureName': featureName,
        'featureOrder': featureSetColumns[featureSet].index(featureName)
    }
    for featureSet, groupNames in featureSetGroups.items()
    for groupName in groupNames
    for featureName in featureGroups[groupName]
])
requiredColumns = sorted(set(sum(featureSetColumns.values(), [])))
assert set(requiredColumns).issubset(mlTable.columns)
assert mlTable.select(pl.sum_horizontal(pl.col(requiredColumns).null_count()).sum()).item() == 0
assert mlTable.select(pl.col('rowId').n_unique()).item() == mlTable.height
assert set(mlTable.get_column('tile').unique().to_list()) == {42, 49, 55}
featureSets


In [ ]:
evaluationRows = (
    mlTable
    .with_columns(
        pl.col('s2Date').cast(pl.Date).alias('targetDate'),
        pl.col('s1Date').cast(pl.Date).alias('radarDate'),
        pl.col('offsetDays').abs().alias('absOffsetDays'),
        (pl.col('ndvi') > 0.95).alias('flagHighNdvi'),
        (pl.col('ndvi') < 0).alias('flagNegativeNdvi'),
        (pl.col('vvDb') > -3).alias('flagBrightVv')
    )
    .with_columns(
        pl.any_horizontal('flagHighNdvi', 'flagNegativeNdvi', 'flagBrightVv').alias('flaggedObservation'),
        pl.when(pl.col('ndvi') < 0).then(pl.lit('<0'))
        .when(pl.col('ndvi') < 0.2).then(pl.lit('0-<0.2'))
        .when(pl.col('ndvi') < 0.4).then(pl.lit('0.2-<0.4'))
        .when(pl.col('ndvi') < 0.6).then(pl.lit('0.4-<0.6'))
        .when(pl.col('ndvi') < 0.8).then(pl.lit('0.6-<0.8'))
        .otherwise(pl.lit('>=0.8')).alias('ndviRange'),
        pl.when(pl.col('absOffsetDays') < 1).then(pl.lit('<1'))
        .when(pl.col('absOffsetDays') < 2).then(pl.lit('1-<2'))
        .otherwise(pl.lit('2-3')).alias('absOffsetRange'),
        pl.when(pl.col('offsetDays') < 0).then(pl.lit('radarBeforeOptical'))
        .when(pl.col('offsetDays') > 0).then(pl.lit('radarAfterOptical'))
        .otherwise(pl.lit('sameTime')).alias('offsetDirection')
    )
    .with_columns(
        pl.col('ndviRange').cast(pl.Categorical),
        pl.col('absOffsetRange').cast(pl.Categorical),
        pl.col('offsetDirection').cast(pl.Categorical)
    )
)
showcaseRows = (
    showcaseApply
    .with_columns(
        pl.lit(42).alias('outerFold'),
        pl.col('s1Date').cast(pl.Date).alias('radarDate'),
        pl.lit('showcaseApply').alias('sourceTable')
    )
)
evaluationRows.group_by('tile').agg(pl.len().alias('rowCount'), pl.col('blockId').n_unique().alias('blockCount'), pl.col('window').n_unique().alias('windowCount')).sort('tile')


In [ ]:
outerFolds = pl.DataFrame({'outerFold': [42, 49, 55]})
models = pl.DataFrame({'model': ['rf', 'xgb'], 'tuningConfigurationCount': [18, 12]})
seeds = pl.DataFrame({'seed': list(range(5))})
featureSetNames = pl.DataFrame({'featureSet': list(featureSetColumns)})
outerFoldAssignments = (
    evaluationRows.select('rowId', 'blockId', 'tile', 'window', 'role')
    .join(outerFolds, how='cross')
    .with_columns(
        pl.when(pl.col('tile') == pl.col('outerFold')).then(pl.lit('test'))
        .when(pl.col('role') == 'pool').then(pl.lit('train'))
        .otherwise(pl.lit('excluded')).alias('splitRole')
    )
    .sort('outerFold', 'rowId')
)
primaryRuns = (
    outerFolds
    .join(models, how='cross')
    .join(featureSetNames, how='cross')
    .join(seeds, how='cross')
    .with_columns(
        pl.lit('window').alias('innerGroup'),
        pl.lit(30).alias('maximumTuningConfigurations'),
        pl.lit('mae').alias('tuningMetric')
    )
    .sort('outerFold', 'model', 'featureSet', 'seed')
)
metricCatalog = pl.DataFrame({
    'metric': ['mae', 'rmse', 'r2', 'bias'],
    'direction': ['lower', 'lower', 'higher', 'zero'],
    'aggregationUnit': ['row', 'row', 'row', 'row'],
    'definition': ['meanAbsError', 'rootMeanSquaredError', 'coefficientOfDetermination', 'meanPredictionMinusObservation']
})
outerFoldAssignments.group_by('outerFold', 'splitRole').len().sort('outerFold', 'splitRole'), primaryRuns.shape


In [ ]:
opticalObservations = (
    s2Observations
    .filter(pl.col('ndvi').is_not_null(), pl.col('validFrac') >= 0.8)
    .select(
        'blockId',
        pl.col('date').cast(pl.Date).alias('opticalDate'),
        pl.col('ndvi').alias('opticalNdvi')
    )
    .unique(['blockId', 'opticalDate'])
    .sort('opticalDate', 'blockId')
)
gapTargets = (
    evaluationRows
    .select('rowId', 'blockId', 'tile', 'window', 'role', 'targetDate', 'ndvi')
    .sort('targetDate', 'blockId')
)
previousOptical = opticalObservations.rename({'opticalDate': 'previousOpticalDate', 'opticalNdvi': 'previousOpticalNdvi'})
nextOptical = opticalObservations.rename({'opticalDate': 'nextOpticalDate', 'opticalNdvi': 'nextOpticalNdvi'})
opticalGapCases = (
    gapTargets
    .join_asof(
        previousOptical,
        left_on='targetDate',
        right_on='previousOpticalDate',
        by='blockId',
        strategy='backward',
        allow_exact_matches=False
    )
    .join_asof(
        nextOptical,
        left_on='targetDate',
        right_on='nextOpticalDate',
        by='blockId',
        strategy='forward',
        allow_exact_matches=False
    )
    .with_columns(
        (pl.col('targetDate') - pl.col('previousOpticalDate')).dt.total_days().alias('gapAgeDays'),
        (pl.col('nextOpticalDate') - pl.col('targetDate')).dt.total_days().alias('daysToNextOptical'),
        (pl.col('nextOpticalDate') - pl.col('previousOpticalDate')).dt.total_days().alias('bracketingGapDays'),
        pl.col('previousOpticalNdvi').alias('locfNdvi'),
        pl.col('previousOpticalDate').is_not_null().alias('locfEvaluable'),
        pl.all_horizontal(pl.col('previousOpticalDate').is_not_null(), pl.col('nextOpticalDate').is_not_null()).alias('interpolationEvaluable')
    )
    .with_columns(
        (pl.col('gapAgeDays') / pl.col('bracketingGapDays')).alias('interpolationWeight')
    )
    .with_columns(
        (pl.col('previousOpticalNdvi') + pl.col('interpolationWeight') * (pl.col('nextOpticalNdvi') - pl.col('previousOpticalNdvi'))).alias('interpolationNdvi'),
        pl.when(pl.col('gapAgeDays') <= 5).then(pl.lit('<=5'))
        .when(pl.col('gapAgeDays') <= 10).then(pl.lit('6-10'))
        .when(pl.col('gapAgeDays') <= 20).then(pl.lit('11-20'))
        .when(pl.col('gapAgeDays') <= 30).then(pl.lit('21-30'))
        .otherwise(pl.lit('>30')).alias('locfGapRange'),
        pl.when(pl.col('bracketingGapDays') <= 10).then(pl.lit('<=10'))
        .when(pl.col('bracketingGapDays') <= 20).then(pl.lit('11-20'))
        .when(pl.col('bracketingGapDays') <= 30).then(pl.lit('21-30'))
        .when(pl.col('bracketingGapDays') <= 45).then(pl.lit('31-45'))
        .when(pl.col('bracketingGapDays') <= 60).then(pl.lit('46-60'))
        .otherwise(pl.lit('>60')).alias('interpolationGapRange')
    )
    .sort('rowId')
)
opticalGapCases.select(pl.sum('locfEvaluable').alias('locfCases'), pl.sum('interpolationEvaluable').alias('interpolationCases'))


In [ ]:
poolRows = evaluationRows.filter(pl.col('role') == 'pool').select('rowId', 'blockId', 'tile', 'window')
randomAssignments = []
groupedAssignments = []
for splitSeed in range(5):
    randomAssignments.append(
        poolRows
        .with_columns(
            (pl.struct('rowId').hash(seed=splitSeed) % 5).cast(pl.Int8).alias('fold'),
            pl.lit('random').alias('splitScheme'),
            pl.lit(splitSeed).alias('splitSeed')
        )
    )
    windowAssignments = (
        poolRows
        .select('window')
        .unique()
        .with_columns(pl.col('window').hash(seed=splitSeed).alias('windowHash'))
        .sort('windowHash')
        .with_row_index('windowOrder')
        .with_columns((pl.col('windowOrder') % 5).cast(pl.Int8).alias('fold'))
        .select('window', 'fold')
    )
    groupedAssignments.append(
        poolRows
        .join(windowAssignments, on='window', how='left')
        .with_columns(
            pl.lit('groupedWindow').alias('splitScheme'),
            pl.lit(splitSeed).alias('splitSeed')
        )
    )
spatialAssignments = (
    poolRows
    .with_columns(
        pl.col('tile').replace_strict({42: 0, 49: 1, 55: 2}).cast(pl.Int8).alias('fold'),
        pl.lit('spatialTile').alias('splitScheme'),
        pl.lit(0).alias('splitSeed')
    )
)
reviewerSplitAssignments = (
    pl.concat(randomAssignments + groupedAssignments + [spatialAssignments])
    .select('rowId', 'blockId', 'tile', 'window', 'splitScheme', 'splitSeed', 'fold')
    .sort('splitScheme', 'splitSeed', 'fold', 'rowId')
)
reviewerSplitAssignments.group_by('splitScheme', 'splitSeed', 'fold').agg(pl.len().alias('rowCount'), pl.col('window').n_unique().alias('windowCount')).sort('splitScheme', 'splitSeed', 'fold')


In [ ]:
experimentCatalog = pl.DataFrame([
    {'experimentId': 1, 'experimentName': 'rfVsXgbLeaveOneTileOut', 'experimentGroup': 'primary', 'preparedTable': 'primaryRuns'},
    {'experimentId': 2, 'experimentName': 'featureSetAblation', 'experimentGroup': 'primary', 'preparedTable': 'featureSets'},
    {'experimentId': 3, 'experimentName': 'fiveSeeds', 'experimentGroup': 'primary', 'preparedTable': 'primaryRuns'},
    {'experimentId': 4, 'experimentName': 'fourMetrics', 'experimentGroup': 'primary', 'preparedTable': 'metricCatalog'},
    {'experimentId': 5, 'experimentName': 'radarVsInterpolationByGap', 'experimentGroup': 'operational', 'preparedTable': 'opticalGapCases'},
    {'experimentId': 6, 'experimentName': 'radarVsLocfByGap', 'experimentGroup': 'operational', 'preparedTable': 'opticalGapCases'},
    {'experimentId': 7, 'experimentName': 'gapLengthCrossover', 'experimentGroup': 'operational', 'preparedTable': 'opticalGapCases'},
    {'experimentId': 8, 'experimentName': 'heldOutRegionPerformance', 'experimentGroup': 'robustness', 'preparedTable': 'evaluationRows'},
    {'experimentId': 9, 'experimentName': 'errorByNdviRange', 'experimentGroup': 'robustness', 'preparedTable': 'evaluationRows'},
    {'experimentId': 10, 'experimentName': 'errorBySensorOffset', 'experimentGroup': 'robustness', 'preparedTable': 'evaluationRows'},
    {'experimentId': 11, 'experimentName': 'flaggedObservationSensitivity', 'experimentGroup': 'robustness', 'preparedTable': 'evaluationRows'},
    {'experimentId': 12, 'experimentName': 'splitStrategyComparison', 'experimentGroup': 'optional', 'preparedTable': 'reviewerSplitAssignments'}
])
outputTables = {
    'evaluationRows': evaluationRows,
    'showcaseRows': showcaseRows,
    'featureSets': featureSets,
    'outerFoldAssignments': outerFoldAssignments,
    'primaryRuns': primaryRuns,
    'metricCatalog': metricCatalog,
    'opticalGapCases': opticalGapCases,
    'reviewerSplitAssignments': reviewerSplitAssignments,
    'experimentCatalog': experimentCatalog
}
for tableName, tableFrame in outputTables.items():
    tableFrame.write_parquet(outputPath / f'{tableName}.parquet')
pl.DataFrame({
    'tableName': list(outputTables),
    'rowCount': list(map(lambda tableName: outputTables[tableName].height, outputTables)),
    'columnCount': list(map(lambda tableName: outputTables[tableName].width, outputTables))
}).sort('tableName')


In [ ]:
assert evaluationRows.height == 792
assert evaluationRows.select(pl.col('blockId').n_unique()).item() == 43
assert evaluationRows.filter(pl.col('flaggedObservation')).height == 14
assert outerFoldAssignments.filter(pl.col('splitRole') == 'test').height == evaluationRows.height
assert outerFoldAssignments.filter(pl.col('splitRole') == 'train', pl.col('role') != 'pool').is_empty()
assert primaryRuns.height == 120
assert primaryRuns.select(pl.col('seed').n_unique()).item() == 5
assert primaryRuns.select(pl.col('featureSet').n_unique()).item() == 4
assert opticalGapCases.filter(pl.col('locfEvaluable')).height == 754
assert opticalGapCases.filter(pl.col('interpolationEvaluable')).height == 716
assert reviewerSplitAssignments.filter(pl.col('splitScheme') == 'groupedWindow').group_by('window', 'splitSeed').agg(pl.col('fold').n_unique().alias('foldCount')).filter(pl.col('foldCount') != 1).is_empty()
assert reviewerSplitAssignments.filter(pl.col('splitScheme') == 'spatialTile').group_by('tile').agg(pl.col('fold').n_unique().alias('foldCount')).filter(pl.col('foldCount') != 1).is_empty()
assert experimentCatalog.height == 12
assert all((outputPath / f'{tableName}.parquet').exists() for tableName in outputTables)
pl.DataFrame({
    'checkName': ['evaluationRows', 'flaggedRows', 'locfCases', 'interpolationCases', 'primaryRuns', 'experiments'],
    'value': [evaluationRows.height, evaluationRows.filter(pl.col('flaggedObservation')).height, opticalGapCases.filter(pl.col('locfEvaluable')).height, opticalGapCases.filter(pl.col('interpolationEvaluable')).height, primaryRuns.height, experimentCatalog.height]
})
